In [ ]:
import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments
import re 
from datasets import Dataset

In [ ]:
# Data Loading 
train_dataset = pd.read_csv(r"F:\Documents\ML & AI\Projects\Transformer-Based Text Summarization System\data\samsum-train.csv")
val_dataset = pd.read_csv(r"F:\Documents\ML & AI\Projects\Transformer-Based Text Summarization System\data\samsum-validation.csv")

In [ ]:
print(f"Train Dataset Shape : {train_dataset.shape}")
print(f"Validation Dataset Shape : {val_dataset.shape}")

In [ ]:
train_dataset = train_dataset.drop(columns="id").sample(n=5000 , random_state=42).reset_index(drop=True)

val_dataset = val_dataset.drop(columns="id").sample(n=818 , random_state=42).reset_index(drop=True)

train_dataset = train_dataset.fillna("")
train_dataset = train_dataset.fillna("")
val_dataset = val_dataset.fillna("")
val_dataset = val_dataset.fillna("")

In [ ]:
train_dataset.shape

In [ ]:
train_dataset["dialogue"][2]

In [ ]:
# Cleaning text 

def clean_text(text): 

    text = re.sub(r"\r\n\ " , " " , text)
    text = re.sub(r"\s+" , " " , text)
    text = re.sub(r"<.*?>" , "" , text)
    text = text.lower()


    return text 

train_dataset["dialogue"] = train_dataset["dialogue"].apply(clean_text)
train_dataset["summary"] = train_dataset["summary"].apply(clean_text)
val_dataset["dialogue"] = val_dataset["dialogue"].apply(clean_text)
val_dataset["summary"] = val_dataset["summary"].apply(clean_text)

In [ ]:
train_dataset["dialogue"][2]

In [ ]:
train_data = Dataset.from_pandas(train_dataset)
val_data = Dataset.from_pandas(val_dataset)

In [ ]:
# Tokenizing 
tokenizer = T5Tokenizer.from_pretrained("t5-large")

def preprocess_function(examples):
    inputs = tokenizer(examples["dialogue"], max_length = 512, padding = "max_length", truncation = True)
    outputs = tokenizer(examples["summary"], max_length = 175, padding = "max_length", truncation = True)
    inputs["labels"]= outputs["input_ids"]

    return inputs

train_dataset_tokenized = train_data.map(preprocess_function, batched=True)
val_dataset_tokenized = val_data.map(preprocess_function, batched=True)



In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-large")

training_args = TrainingArguments(
    output_dir=r"F:\Documents\ML & AI\Projects\Transformer-Based Text Summarization System\saved_files",
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    per_device_train_batch_size=2, 
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, 
    fp16=True,  
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    no_cuda=False,  
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=val_dataset_tokenized,
)

# بدء التدريب على الـ GPU
trainer.train()

In [ ]:
model.save_pretrained(r"F:\Documents\ML & AI\Projects\Transformer-Based Text Summarization System\saved_files")
tokenizer.save_pretrained(r"F:\Documents\ML & AI\Projects\Transformer-Based Text Summarization System\saved_files")